In [1]:
import pandas as pd
import numpy as np

In [15]:
real_pm = pd.read_csv('/Users/tamsopanat/Desktop/MedCMU - FamMed/Dustboy/result/pm25.csv')
real_temp = pd.read_csv('./data/temp.csv')
real_temp.loc[real_temp['amphoe'].str.contains('เมือง', na=False), 'amphoe'] = 'เมือง'

required_dates = set([
    '2026-05-01', '2026-05-02', '2026-05-03', '2026-05-04', 
    '2026-05-05', '2026-05-06', '2026-05-07'
])

def get_complete_locations(df, req_dates):
    # Group by location and collect all dates into a set
    loc_dates = df.groupby(['province', 'amphoe', 'tambon'])['date'].apply(set).reset_index()
    # Filter for locations where the required_dates are a subset of available dates
    valid = loc_dates[loc_dates['date'].apply(lambda dates: req_dates.issubset(dates))]
    # Return just the address columns
    return valid[['province', 'amphoe', 'tambon']]

# 4. Find valid locations in each dataset independently
valid_temp_locs = get_complete_locations(real_temp, required_dates)
valid_pm_locs = get_complete_locations(real_pm, required_dates)

# 5. Find the strict intersection (locations that are complete in BOTH datasets)
master_valid_locations = pd.merge(
    valid_temp_locs, 
    valid_pm_locs, 
    on=['province', 'amphoe', 'tambon'], 
    how='inner'
)

# 6. Filter the original datasets using this master list of valid locations
real_temp = pd.merge(real_temp, master_valid_locations, on=['province', 'amphoe', 'tambon'], how='inner')
real_pm = pd.merge(real_pm, master_valid_locations, on=['province', 'amphoe', 'tambon'], how='inner')

# 7. (Optional) Filter to only include exact matching dates across both datasets
# This ensures that if real_temp has data for May 10th but real_pm doesn't, May 10th is dropped from both.
shared_keys = pd.merge(
    real_temp[['province', 'amphoe', 'tambon', 'date']], 
    real_pm[['province', 'amphoe', 'tambon', 'date']], 
    on=['province', 'amphoe', 'tambon', 'date'], 
    how='inner'
).drop_duplicates()

real_temp = pd.merge(real_temp, shared_keys, on=['province', 'amphoe', 'tambon', 'date'], how='inner')
real_pm = pd.merge(real_pm, shared_keys, on=['province', 'amphoe', 'tambon', 'date'], how='inner')

In [30]:
real = real_pm.merge(real_temp, how='inner', on=['datetime', 'date', 'hour', 'province', 'amphoe', 'tambon'])
mean_temps = real_temp.groupby(by=['datetime', 'date', 'hour'], as_index=False)['temperature'].mean()
mean_temps = mean_temps.rename(columns={'temperature': 'hourly_mean_temp'})
real = real.merge(mean_temps, how='left', on=['datetime', 'date', 'hour'])
real['temperature'] = real['temperature'].fillna(real['hourly_mean_temp'])
real = real.drop(columns=['hourly_mean_temp'])
real = real.rename(columns={'pm25': 'pm25_avg', 'temperature': 'temperature_avg'})
real.drop(columns = ['date', 'hour'], inplace = True)
real.rename(columns = {'datetime' : 'date'}, inplace = True)
real['local_name'] = real['local_name'].str.split(' ต.').str[0]
real[real['date'] >= '2026-04-01 00:00:00'].to_csv('./environmental.csv', index= False)
real

,date,province,amphoe,tambon,local_name,pm25_avg,temperature_avg
0,2026-01-01 00:00:00,เชียงใหม่,ฝาง,เวียง,รพ.ฝาง,72.90,15.50
1,2026-01-01 01:00:00,เชียงใหม่,ฝาง,เวียง,รพ.ฝาง,71.09,14.93
2,2026-01-01 02:00:00,เชียงใหม่,ฝาง,เวียง,รพ.ฝาง,78.88,14.48
3,2026-01-01 03:00:00,เชียงใหม่,ฝาง,เวียง,รพ.ฝาง,76.91,14.39
4,2026-01-01 04:00:00,เชียงใหม่,ฝาง,เวียง,รพ.ฝาง,61.65,13.77
...,...,...,...,...,...,...,...
154436,2026-06-30 19:00:00,เชียงใหม่,เชียงดาว,เชียงดาว,หน่วยพิทักษ์ป่าสบห้วยผาตั้ง-นาเลา,6.57,27.89
154437,2026-06-30 20:00:00,เชียงใหม่,เชียงดาว,เชียงดาว,หน่วยพิทักษ์ป่าสบห้วยผาตั้ง-นาเลา,6.30,27.04
154438,2026-06-30 21:00:00,เชียงใหม่,เชียงดาว,เชียงดาว,หน่วยพิทักษ์ป่าสบห้วยผาตั้ง-นาเลา,8.15,26.58
154439,2026-06-30 22:00:00,เชียงใหม่,เชียงดาว,เชียงดาว,หน่วยพิทักษ์ป่าสบห้วยผาตั้ง-นาเลา,9.76,26.08


In [4]:
import csv
import random
import math
from datetime import datetime, timedelta

locations = {}
for province, amphoe, tambon in real[['province', 'amphoe', 'tambon']].drop_duplicates().itertuples(index=False):
    locations.setdefault(province, {}).setdefault(amphoe, []).append(tambon)

start_date = datetime(2026, 4, 1, 0, 0, 0)
end_date = datetime(2026, 6, 30, 23, 0, 0)
num_days = (end_date - start_date).days
num_hours = num_days * 24


pat_filename = "patients.csv"
pat_headers = [
    "patient_id", "name_masked", "sex", "age", "province", "amphoe", "tambon", "last_contact_date",
    "age_over_65", "age_under_5", "pregnant", "bedridden_immobile", "outdoor_worker", 
    "copd", "asthma", "cardiovascular_disease", "diabetes", "ckd", "hypertension", "lung_cancer", "post_covid"
]

thai_first_names = ["Somchai", "Mali", "Anong", "Kittipong", "Siriporn", "Nattapong", "Wipawan", "Surasak", "Pornthip", "Arunee"]
thai_last_initials = ["P.", "S.", "W.", "K.", "T.", "N.", "J.", "M."]

with open(pat_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(pat_headers)
    
    # Generate 1500 mock patients to fill the expanded province/amphoe list
    for i in range(1, 2000):
        pid = f"PID-{random.randint(1000, 9999)}"
        name = f"{random.choice(thai_first_names)} {random.choice(thai_last_initials)}"
        sex = random.choice(["M", "F"])
        age = random.randint(2, 90)
        
        # Select random location from hierarchy
        prov = random.choice(list(locations.keys()))
        amphoe = random.choice(list(locations[prov].keys()))
        tambon = random.choice(locations[prov][amphoe])
        
        # Generate a recent contact date
        contact_days_ago = random.randint(1, 60)
        contact_date = (start_date + timedelta(days=num_days) - timedelta(days=contact_days_ago)).strftime("%Y-%m-%d")
        
        # Demographics booleans
        age_over_65 = age > 65
        age_under_5 = age < 5
        pregnant = sex == "F" and 18 <= age <= 40 and random.random() < 0.05
        bedridden_immobile = age_over_65 and random.random() < 0.2
        outdoor_worker = 18 <= age <= 60 and random.random() < 0.3
        
        # Health conditions booleans (weighted by age roughly)
        age_factor = age / 100
        copd = random.random() < (0.1 + age_factor * 0.3)
        asthma = random.random() < 0.15
        cvd = random.random() < (0.05 + age_factor * 0.4)
        diabetes = random.random() < (0.05 + age_factor * 0.3)
        ckd = random.random() < (0.02 + age_factor * 0.2)
        hypertension = random.random() < (0.1 + age_factor * 0.5)
        lung_cancer = random.random() < (0.01 + age_factor * 0.1)
        post_covid = random.random() < 0.2
        
        writer.writerow([
            pid, name, sex, age, prov, amphoe, tambon, contact_date,
            age_over_65, age_under_5, pregnant, bedridden_immobile, outdoor_worker,
            copd, asthma, cvd, diabetes, ckd, hypertension, lung_cancer, post_covid
        ])

print(f"✅ Generated {pat_filename} successfully.")

✅ Generated patients.csv successfully.
